# Exploratory Data Analysis — Breast Cancer Dataset

**Dataset:** Breast Cancer Wisconsin (`sklearn`) — 569 samples, 30 features, binary target
**Goal:** Understand the data structure, check quality, and identify which features separate malignant from benign diagnoses before modeling.

This mirrors the EDA steps from *Data Mining for Business Analytics* (Shmueli et al.), Chapter 3, implemented in Python.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer

sns.set_theme(style="whitegrid", palette="muted")
np.random.seed(42)
print("Libraries loaded.")

## 1. Load the Dataset

In [ ]:
raw = load_breast_cancer(as_frame=True)
df = raw.frame
df['diagnosis'] = df['target'].map({0: 'Malignant', 1: 'Benign'})

print(f"Shape: {df.shape}")
print(f"Features: {list(raw.feature_names[:5])} ...")
df.head()

## 2. Data Quality Check

In [ ]:
print("Data types:")
print(df.dtypes.value_counts())
print(f"\nMissing values: {df.isnull().sum().sum()} (none expected for sklearn built-ins)")
print(f"\nClass distribution:")
print(df['diagnosis'].value_counts())
print(f"\nImbalance ratio: "
      f"{df['diagnosis'].value_counts()[0] / df['diagnosis'].value_counts()[1]:.2f}:1 "
      f"(malignant:benign)")

## 3. Class Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = df['diagnosis'].value_counts()
colors = ['#e74c3c', '#2ecc71']
bars = ax.bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.5)
for bar, count in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            str(count), ha='center', va='bottom', fontweight='bold')
ax.set_title('Class Distribution — Breast Cancer Diagnosis', fontsize=14, pad=12)
ax.set_ylabel('Count')
ax.set_ylim(0, counts.max() * 1.15)
plt.tight_layout()
plt.show()

## 4. Feature Distributions by Diagnosis

In [ ]:
features = raw.feature_names[:6].tolist()
palette = {'Malignant': '#e74c3c', 'Benign': '#2ecc71'}

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flatten(), features):
    for label, color in palette.items():
        ax.hist(df[df['diagnosis'] == label][col], bins=25,
                alpha=0.6, color=color, label=label, edgecolor='white')
    ax.set_title(col, fontsize=9)
    ax.legend(fontsize=7)
fig.suptitle('Feature Distributions by Diagnosis (First 6 Features)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 5. Correlation Heatmap (Mean Features)

In [ ]:
mean_cols = [c for c in df.columns if 'mean' in c and c != 'target']
corr = df[mean_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0,
            annot=True, fmt='.2f', annot_kws={'size': 7},
            linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Heatmap — Mean Features', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Which Features Best Predict the Diagnosis?

In [ ]:
corr_target = (df.drop(columns=['diagnosis'])
                 .corr()['target']
                 .drop('target')
                 .sort_values())

fig, ax = plt.subplots(figsize=(8, 12))
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in corr_target]
corr_target.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title(
    'Feature Correlation with Target\n(positive = Benign, negative = Malignant)',
    fontsize=12
)
ax.set_xlabel('Pearson Correlation')
plt.tight_layout()
plt.show()

## Key Findings

- **No missing values** — the sklearn version is pre-cleaned.
- **Moderate class imbalance** (~1.68:1 benign:malignant) — manageable with `class_weight='balanced'`.
- **Worst-case features** (`worst radius`, `worst perimeter`, `worst area`) are most strongly correlated with diagnosis, followed by mean radius and mean concave points.
- **High multi-collinearity** among size-related features (radius, perimeter, area). Regularized models (Ridge, Lasso) or dimensionality reduction would help.
- **Clear class separation** visible in radius and concavity features — good signal for a linear classifier.